<a href="https://colab.research.google.com/github/SarahkhIT/DataEngProject/blob/main/notebooks/05_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5. RAG Pipeline — Retrieval & Generation

Chunking → ChromaDB embeddings + BM25 → Reciprocal Rank Fusion hybrid search → cross-encoder reranking → cited, grounded answer.

In [ ]:
## Load papers & build documents
papers_df = pd.read_csv("arxiv_valid_bronze.csv", dtype=str)

required_columns = ["paper_id", "title", "abstract", "authors", "category", "published_date"]
missing_columns = [c for c in required_columns if c not in papers_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

papers_df = papers_df[required_columns].copy()
for column in required_columns:
    papers_df[column] = papers_df[column].fillna("").astype(str).str.strip()

print(f"✅ Loaded {len(papers_df)} valid papers.")
papers_df.head()

✅ Loaded 1500 valid papers.


,paper_id,title,abstract,authors,category,published_date
0,0704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-pe...",Ileana Streinu and Louis Theran,cs.CG,2008-12-13
1,0704.0022,Stochastic Lie group integrators,We present Lie group integrators for nonlinear...,Simon J.A. Malham and Anke Wiese,cs.NA,2026-06-03
2,0704.0046,A limit relation for entropy and channel capac...,"In a quantum mechanical model, Diosi, Feldmann...","I. Csiszar, F. Hiai and D. Petz",cs.IT,2009-11-13
3,0704.0047,Intelligent location of simultaneously active ...,The intelligent acoustic emission locator is d...,T. Kosel and I. Grabec,cs.NE,2009-09-29
4,0704.0050,Intelligent location of simultaneously active ...,Part I describes an intelligent acoustic emiss...,T. Kosel and I. Grabec,cs.NE,2007-05-23


In [ ]:
DOCUMENTS = []
for _, paper in papers_df.iterrows():
    DOCUMENTS.append({
        "id": paper["paper_id"],
        "text": paper["abstract"],
        "title": paper["title"],
        "authors": paper["authors"],
        "category": paper["category"],
        "published_date": paper["published_date"]
    })

print(f"✅ Converted {len(DOCUMENTS)} papers into RAG documents.")

✅ Converted 1500 papers into RAG documents.


#Chunking

In [ ]:
import re

def split_into_sentences(text):
    text = str(text).strip()
    if not text:
        return []
    return re.split(r"(?<=[.!?])\s+", text)

def chunk_documents(documents, chunk_size=2, overlap=1):
    chunks = []
    for document in documents:
        sentences = split_into_sentences(document["text"])
        for start in range(0, len(sentences), chunk_size - overlap or 1):
            piece = sentences[start:start + chunk_size]
            if not piece:
                continue
            chunks.append({
                "chunk_id": f"{document['id']}_{start}",
                "paper_id": document["id"],
                "title": document["title"],
                "authors": document["authors"],
                "category": document["category"],
                "published_date": document["published_date"],
                "text": " ".join(piece),
            })
    return chunks

chunks = chunk_documents(DOCUMENTS, chunk_size=2, overlap=1)
print(f"✅ {len(DOCUMENTS)} papers became {len(chunks)} chunks.")

✅ 1500 papers became 8781 chunks.


## Vector store — ChromaDB

In [ ]:
from chromadb import Client
from chromadb.config import Settings
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

embedding_function = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client_chroma = Client(Settings())
collection = client_chroma.get_or_create_collection(name="arxiv_rag", embedding_function=embedding_function)
print("✅ ChromaDB collection created.")

✅ ChromaDB collection created.


In [ ]:
def add_chunks_to_chroma(collection, chunks, batch_size=1000):
    total = len(chunks)

    for start in range(0, total, batch_size):
        batch = chunks[start:start + batch_size]

        collection.add(
            ids=[chunk["chunk_id"] for chunk in batch],
            documents=[chunk["text"] for chunk in batch],
            metadatas=[
                {
                    "paper_id": chunk["paper_id"],
                    "title": chunk["title"],
                    "authors": chunk["authors"],
                    "category": chunk["category"],
                    "published_date": chunk["published_date"],
                }
                for chunk in batch
            ],
        )

        print(f"Added {min(start + batch_size, total)} / {total} chunks")

add_chunks_to_chroma(collection, chunks, batch_size=1000)
print(f"ChromaDB insert complete: {len(chunks)} chunks added.")

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Added 1000 / 8781 chunks


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Added 2000 / 8781 chunks


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

...

2026-07-22T23:16:12.015758Z [info     ] Updated metadata: ClusterMetadata(brokers: 1, topics: 1, coordinators: 1) [kafka.cluster] loc=cluster.py:638
2026-07-22T23:16:12.669381Z [info     ] Updated metadata: ClusterMetadata(brokers: 1, topics: 1, coordinators: 0) [kafka.cluster] loc=cluster.py:638
Added 7000 / 8781 chunks


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Added 8000 / 8781 chunks


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Added 8781 / 8781 chunks
ChromaDB insert complete: 8781 chunks added.


## Keyword search — BM25

In [ ]:
from rank_bm25 import BM25Okapi

class BM25Index:
    def __init__(self, chunks):
        self.chunks = chunks
        self.tokenized_documents = [chunk["text"].lower().split() for chunk in chunks]
        self.index = BM25Okapi(self.tokenized_documents)

    def search(self, query, top_k=5):
        tokenized_query = query.lower().split()
        scores = self.index.get_scores(tokenized_query)
        ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        return [
            {**self.chunks[i], "chunk_id": self.chunks[i]["chunk_id"], "bm25_score": float(scores[i])}
            for i in ranked_indices
        ]

bm25_index = BM25Index(chunks)
print("✅ BM25 index created successfully.")

✅ BM25 index created successfully.


## Hybrid search — Reciprocal Rank Fusion

In [ ]:
def reciprocal_rank_fusion(vector_hits, bm25_hits, k=60, top_k=10):
    scores, combined_results = {}, {}

    for rank, result in enumerate(vector_hits, start=1):
        chunk_id = result["chunk_id"]
        scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (k + rank)
        combined_results[chunk_id] = result

    for rank, result in enumerate(bm25_hits, start=1):
        chunk_id = result["chunk_id"]
        scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (k + rank)
        combined_results.setdefault(chunk_id, result)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [{**combined_results[cid], "rrf_score": score} for cid, score in ranked]

def hybrid_search(query, top_k=10):
    vector = collection.query(query_texts=[query], n_results=top_k, include=["documents", "metadatas", "distances"])
    vector_hits = []
    for i in range(len(vector["ids"][0])):
        metadata = vector["metadatas"][0][i]
        vector_hits.append({
            "chunk_id": vector["ids"][0][i],
            "paper_id": metadata["paper_id"], "title": metadata["title"],
            "authors": metadata["authors"], "category": metadata["category"],
            "published_date": metadata["published_date"], "text": vector["documents"][0][i],
        })

    bm25_hits = bm25_index.search(query, top_k=top_k)
    return reciprocal_rank_fusion(vector_hits, bm25_hits, top_k=top_k)

results = hybrid_search("machine learning", top_k=5)
for r in results:
    print("=" * 60); print("Title:", r["title"]); print("RRF Score:", r["rrf_score"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Title: An Adaptive Strategy for the Classification of G-Protein Coupled   Receptors
RRF Score: 0.01639344262295082
Title: An Adaptive Strategy for the Classification of G-Protein Coupled   Receptors
RRF Score: 0.016129032258064516
Title: An Adaptive Strategy for the Classification of G-Protein Coupled   Receptors
RRF Score: 0.015873015873015872
Title: Ensemble Learning for Free with Evolutionary Algorithms ?
RRF Score: 0.015625
Title: Learning Phonotactics Using ILP
RRF Score: 0.015384615384615385


## Reranking — cross-encoder

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ CrossEncoder reranker loaded.")

def rerank_results(query, results, top_k=5):
    pairs = [[query, r["text"]] for r in results]
    scores = reranker.predict(pairs)
    reranked = []
    for r, score in zip(results, scores):
        updated = r.copy()
        updated["reranker_score"] = float(score)
        reranked.append(updated)
    reranked.sort(key=lambda r: r["reranker_score"], reverse=True)
    return reranked[:top_k]

2026-07-22T22:59:08.057880Z [info     ] No device provided, using cpu  [sentence_transformers.base.model] loc=model.py:190
2026-07-22T22:59:08.235209Z [info     ] No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model. [sentence_transformers.base.model] loc=model.py:990


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ CrossEncoder reranker loaded.


## RAG prompt with citations

In [ ]:
def build_rag_context(results):
    parts = []
    for i, r in enumerate(results, start=1):
        parts.append(f"""Source [{i}]
Title: {r['title']}
Authors: {r['authors']}
Published Date: {r['published_date']}
Category: {r['category']}
Content: {r['text']}""".strip())
    return "\n\n".join(parts)

def build_rag_prompt(question, results):
    context = build_rag_context(results)
    return f"""You are a research assistant.

Answer the question using only the sources provided below.

Rules:
1. Do not use outside knowledge.
2. If the answer is not supported by the sources, say:
   "The retrieved sources do not provide enough information."
3. Cite every important claim using source numbers such as [1], [2], or [3].
4. At the end, include a Sources section listing the cited paper titles.

Question: {question}

Sources:
{context}"""

In [ ]:
import os
from google.colab import userdata
from groq import Groq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def generate_rag_answer_groq(question, top_k=5):
    hybrid_results = hybrid_search(question, top_k=10)
    reranked_results = rerank_results(question, hybrid_results, top_k=top_k)
    prompt = build_rag_prompt(question, reranked_results)

    chat_completion = groq_client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.1-8b-instant",
        temperature=0.1
    )
    return chat_completion.choices[0].message.content

print("✅ Groq client ready.")

✅ Groq client ready.


### Run it — with lineage tracking

In [ ]:
question_to_ask = "What are the limitations of traditional machine learning tools in bioinformatics?"

answer = run_with_lineage("rag_answer_generation", generate_rag_answer_groq, question_to_ask)

print("=" * 60)
print("RAG GENERATED ANSWER WITH CITATIONS (LLaMA via Groq):")
print("=" * 60)
print(answer)

[OpenLineage] START — rag_answer_generation


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[OpenLineage] COMPLETE — rag_answer_generation
RAG GENERATED ANSWER WITH CITATIONS (LLaMA via Groq):
The limitations of traditional machine learning tools in bioinformatics are that they are unable to accommodate new information into their existing models. This is because many machine learning tools have been applied to this problem using static machine learning structures such as neural networks or support vector machines [1, 2]. These traditional machine learning tools are unable to incrementally learn new data as it becomes available, which is a limitation in bioinformatics where new data is constantly being generated [1, 2].

In contrast, alternative machine learning systems such as fuzzy ARTMAP have the ability to incrementally learn new data as it becomes available [2, 4]. However, the retrieved sources do not provide enough information to determine if this is a limitation of traditional machine learning tools in bioinformatics or a general limitation of traditional machine learn

**RAG complete** — hybrid search, reranking, and cited answer generation all working and lineage-tracked.